[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/05_windows_and_unreadable_recordings.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Window-level calls and unreadable recordings

Allow about 40 minutes. What should make a whole recording enter AF
review? You will turn window calls into recording decisions, measure their
consequences per recording, and price a repeat pathway for a recording nobody
can read. The window recordings are fully synthetic; the repeat pathway
borrows the simulated forest's development sensitivity and specificity.

The action is **send this recording for AF review**, priced as in the core
notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0, in illustrative loss units.
Replace each `...` and run the checker below it; if you are stuck for about
10 minutes, use the stuck switch below the function, where there is one, or open the worked copy in `extension/solutions/`.
A later cell that needs an unfinished problem prints `Waiting for ...`: once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell.

In [ ]:
import numpy as np
from IPython.display import display
from clinical_utility import data, reference, windows
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
dataset = windows.synthetic_window_recordings()
window_positive = dataset["window_scores"] >= .5
truth = dataset["af_present"]

## Match the label to the question

CinC gives one label per recording. Each synthetic recording here has
100 windows in time order, a score for each, and a known set of AF windows.
The windows have no stated length or overlap, so "consecutive" below means
neighbouring window indices, not a duration.

### Problem A: Predict

Can a correct recording-level AF label validate *where* in the recording a
window call placed AF? Store `"yes"` or `"no"`, and write in `missing` which
reference information would be needed.

In [ ]:
localisation_validated = "no"
missing = """
Interval truth: when each AF episode starts and stops. A recording label says only
that AF is present somewhere.
"""
_ = check.run("window_predict", localisation_validated)

A recording label can assess whether a recording containing AF enters review.
It cannot say where AF begins or ends, how much AF there is, or whether a
particular window was called correctly. Copying a recording's label onto every
window does not make a localisation evaluation.

Four rules turn window calls into one decision. Any-positive reviews a
recording after any positive window. Majority asks whether strictly more than
half are positive. A count rule asks for at least `k` positive windows
anywhere. A persistence rule asks for a run of consecutive positive windows.
Take `[True, False, True, False]`: any-positive reviews it and majority does
not. Separated spikes can meet a count rule but not a run rule. Each rule is a
policy about burden or duration; our reference is simply **any AF present in
the recording**.

## Choose the aggregation

### Problem B: Make

Write `aggregate(positive, k)` for a boolean array shaped
`(recordings, windows)`. Return a dictionary of boolean arrays under the keys
`any`, `majority` and `count`, one decision per recording. Majority is strictly
more than half; count includes equality at `k`.

In [ ]:
def aggregate(positive, k):
    positive = np.asarray(positive, dtype=bool)
    total = positive.sum(axis=1)
    return {"any": positive.any(axis=1), "majority": total > positive.shape[1] / 2,
            "count": total >= k}
_ = check.run("window_rules", aggregate)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# aggregate = check.use_reference("aggregate")

In [ ]:
# An artefact recording and one with brief AF, against the synthetic interval truth.
for kind in ("artefact", "paroxysmal"):
    i = np.flatnonzero(dataset["kind"] == kind)[0]
    display(view.windows(dataset["recording"][i], dataset["window_scores"][i],
                         dataset["af_windows"][i], .5))
    print(kind, "AF interval [start, stop):", dataset["af_interval"][i])

At a window cut of 0.5, any-positive makes 108 false-positive recording
reviews, including all 86 artefact recordings. Majority misses all 79 brief
paroxysms. The first treats an isolated spike as enough evidence; the second
discards AF that occupies a minority of windows. These errors come from the
aggregation, not only from the quality of the window scores.

## Require a run

### Problem C: Modify

Write `persistent(positive, run_length)`. Loop over recordings and then over
windows, tracking the current run and resetting it to zero after a negative
window. Return a boolean array with one decision per recording. A sum over all
windows cannot tell an unbroken run from separated artefacts.

In [ ]:
def persistent(positive, run_length):
    result = []
    for row in np.asarray(positive, dtype=bool):
        current = best = 0
        for call in row:
            current = current + 1 if call else 0
            best = max(best, current)
        result.append(best >= run_length)
    return np.array(result, dtype=bool)
_ = check.run("window_persistence", persistent)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# persistent = check.use_reference("persistent")

### Problem D: Make

Write `recording_counts(review, af_present)`, returning a dictionary with the
keys `TP`, `FP`, `FN` and `TN` from one review decision and one reference label
per recording. The checker uses unequal counts, so swapped cells are caught.

In [ ]:
def recording_counts(review, af_present):
    review, af = np.asarray(review, dtype=bool), np.asarray(af_present, dtype=bool)
    return {"TP": int(np.sum(review & af)), "FP": int(np.sum(review & ~af)),
            "FN": int(np.sum(~review & af)), "TN": int(np.sum(~review & ~af))}
_ = check.run("window_counts", recording_counts)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# recording_counts = check.use_reference("recording_counts")

In [ ]:
if check.ready(aggregate=aggregate, persistent=persistent, recording_counts=recording_counts):
    actions = aggregate(window_positive, 3)
    actions["persistence"] = persistent(window_positive, 3)
    rows = []
    for name, review in actions.items():
        counts = recording_counts(review, truth)
        rows.append({"rule": name, **counts,
                     "loss": sum(counts[k] * costs[k] for k in counts),
                     "brief AF reviewed": int(review[dataset["kind"] == "paroxysmal"].sum())})
    display(view.table(list(rows[0]), rows, "Recording-level consequences at the same window cut"))

Persistence with a run of 3 windows gives 0 false positives and reviews
76 of the 79 brief paroxysms. That does not make persistence best in general:
shorter runs, interrupted detections or longer artefacts could reverse the
order. The run length was set for this example, not chosen by optimisation.

**Think:** a run counts window indices. With real ECG windows of length `L`
taken every `s` seconds, a run of `k` windows covers `L + (k - 1) * s`
seconds of recording. That is the span the windows cover, not an established
duration of continuous AF: each window is called on its own, and AF need not
fill it. If the windows overlapped by half, would a run of 3 positive
windows cover a longer or a shorter span than windows that do not overlap?

#### Overlap and the persistence rule

Shorter. Halving the stride for the same window length shortens the span that
the same number of windows covers, so the rule becomes easier to meet: a
briefer episode, or a single artefact spanning two overlapping windows, can
satisfy it. Even without overlap, a run shows positive calls across that span,
not AF throughout it. To keep the same requirement, raise `k` as the stride
falls. State a persistence rule as a covered span in seconds, and translate it
into windows only once the length and stride are fixed.

### Problem E: Fix the bug

A report computes "recording sensitivity" with the function below. Fix it in
place so that it applies `rule` to get one decision per recording and returns
the fraction of AF recordings reviewed.

In [ ]:
def recording_sensitivity(window_positive, af_present, rule):
    """Fraction of AF recordings that the rule sends for review."""
    positive = np.asarray(window_positive, dtype=bool)
    af = np.asarray(af_present, dtype=bool)
    return rule(positive)[af].mean()


_ = check.run("window_sensitivity", recording_sensitivity)

In [ ]:
if check.ready(recording_sensitivity=recording_sensitivity):
    for name, rule in (("any-positive", windows.any_positive), ("persistence", windows.persistence(3))):
        print(name, "recording sensitivity:", recording_sensitivity(window_positive, truth, rule))

## Price the unreadable branch

An unreadable recording has no binary AF reference here: the CinC noise label
is not an AF-negative label. A service still needs an action. Use an assumed
AF probability `p` for this pathway rather than silently treating unreadable
recordings as AF-free. "Accept" below means accept the recording without AF
review; it is not a claim that the person has no AF.

| Action | Branch | Probability | Expected loss of the branch |
|---|---|---|---|
| Review now | | | `p*C_TP + (1-p)*C_FP` |
| Accept now | | | `p*C_FN + (1-p)*C_TN` |
| Repeat, paying `repeat_cost + delay_cost` once | readable | `q` | `p*(Se*C_TP + (1-Se)*C_FN) + (1-p)*((1-Sp)*C_FP + Sp*C_TN)` |
| | still unreadable, so review | `1 - q` | `p*C_TP + (1-p)*C_FP` |

The repeat's expected loss is its one-off cost plus the two branch losses
weighted by their probabilities. Assume that AF persists between recordings,
that readability does not depend on AF, and that readable repeats have the
stated `Se` and `Sp`. These are borrowed from the development forest for
illustration; they were not measured on unreadable recordings.

**Worked example**, with other inputs than the problem uses: AF probability
`p` = 0.1, readability `q` = 0.5, `Se` = 0.8, `Sp` = 0.9, a repeat cost of 0.1 and
no delay cost, with the default prices.

| Branch | Expected loss |
|---|---:|
| Review now | 1.000 |
| Accept now | 0.500 |
| Repeat is readable | 0.270 |
| Repeat is still unreadable, so review | 1.000 |
| Repeat: one-off cost, then the two branches weighted by `q` and `1 - q` | 0.1 + 0.5 × 0.270 + 0.5 × 1.000 = 0.735 |

Here accepting now is cheapest: AF is unlikely, so a miss is rarely paid for.

### Problem F: Make

Write `repeat_losses(p, costs, repeat_cost, q, se, sp, delay_cost)`, returning
a dictionary with `review_now`, `no_review_now`, `repeat`, and the cheapest
key as `best`. Then
change only the readability `q` and write in `why_it_reverses` why the
preferred action changes.

In [ ]:
def repeat_losses(p, costs, repeat_cost, q, se, sp, delay_cost=0.):
    review = p * costs["TP"] + (1 - p) * costs["FP"]
    accept = p * costs["FN"] + (1 - p) * costs["TN"]
    readable = (p * (se * costs["TP"] + (1 - se) * costs["FN"])
                + (1 - p) * ((1 - sp) * costs["FP"] + sp * costs["TN"]))
    repeat = repeat_cost + delay_cost + q * readable + (1 - q) * review
    losses = {"review_now": review, "no_review_now": accept, "repeat": repeat}
    return {**losses, "best": min(losses, key=losses.get)}


why_it_reverses = """
When few repeats are readable, most recordings pay for the repeat and then reach
the review fallback anyway, so reviewing at once is cheaper.
"""
_ = check.run("window_repeat", repeat_losses)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# repeat_losses = check.use_reference("repeat_losses")

In [ ]:
# These inputs are assumptions for the repeat pathway, not outcomes from the window data.
base = reference.best_thresholds(reference.sweep_thresholds(
    data.load_records("random_forest", "development"), costs))[0]
se, sp = base["sensitivity"], base["specificity"]
if check.ready(repeat_losses=repeat_losses):
    for p in (.05, .1, .3):
        print("AF probability", p, repeat_losses(p, costs, .2, .8, se, sp))
    print("Only readability changed:", repeat_losses(.3, costs, .2, .2, se, sp))
    print("Only delay cost changed:", repeat_losses(.3, costs, .2, .8, se, sp, delay_cost=.2))

At an assumed AF probability of 30%, a repeat costs 0.893, reviewing now
1.000 and accepting without review 1.500, with readability 80% and a repeat cost of
0.2. Lowering only the readability to 20% raises the repeat's loss to
1.123, and reviewing now becomes cheaper. A delay cost can also remove the
repeat's advantage. State which assumption changed, rather than changing the
prices to explain the result.

## Before you finish

- Keep the recording label distinct from interval truth and AF burden.
- Explain why a count rule and a persistence rule disagree.
- Price the unreadable fallback as well as the readable branch.
- Identify the assumption that reverses repeat versus immediate review.

**Discuss:** If recordings are more often unreadable during AF, how would you
change the repeat tree, and what data would give its branch probabilities?